# NBA Database Quick Start

Get started exploring the NBA database. This notebook demonstrates loading data via
Polars (Parquet) and DuckDB, basic queries, and simple visualizations.

In [ ]:
import polars as pl

df = pl.read_parquet("../nbadb/parquet/analytics_player_game_complete/")
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
df.head(5)

In [ ]:
# Top 20 scorers in 2024-25
top_scorers = (
    df.filter(pl.col("season_id") == "2024-25")
    .group_by("player_name")
    .agg(
        pl.col("pts").mean().alias("ppg"),
        pl.col("reb").mean().alias("rpg"),
        pl.col("ast").mean().alias("apg"),
        pl.col("game_id").n_unique().alias("games"),
    )
    .filter(pl.col("games") >= 20)
    .sort("ppg", descending=True)
    .head(20)
)
top_scorers

In [ ]:
import duckdb

conn = duckdb.connect("../nbadb/nba.duckdb", read_only=True)

# Season leaders via DuckDB
season_leaders = conn.sql("""
    SELECT *
    FROM analytics_player_season_complete
    WHERE season_id = '2024-25'
    ORDER BY pts DESC
    LIMIT 20
""").pl()

print(f"Season leaders: {len(season_leaders)} rows")
season_leaders

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 6))
    data = top_scorers.head(10)
    ax.barh(
        data["player_name"].to_list()[::-1],
        data["ppg"].to_list()[::-1],
        color="steelblue",
    )
    ax.set_xlabel("Points Per Game")
    ax.set_title("Top 10 Scorers (2024-25)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping visualization")
    print("Install with: pip install matplotlib")

In [ ]:
conn.close()

## Next Steps

- **Analytics Deep Dive**: See `analytics_deep_dive.ipynb` for advanced analysis
- **Shot Charts**: Query `fact_shot_chart` for spatial shooting data
- **Play-by-Play**: Query `fact_play_by_play` for possession-level analysis
- **Documentation**: Visit the [nbadb docs](https://nbadb.dev) for full schema reference
- **Dataset**: [Kaggle dataset page](https://www.kaggle.com/datasets/wyattowalsh/basketball)